### NIH Alzheimer's Research Funding vs State-Level Disease Burden

### Research Question

At the U.S. state level (2018-2023), how is NIH Alzheimer's research funding associated with state-level Alzheimer's burden (age-adjusted mortality rates, 65+ and 85+ populations), after controlling for state research capacity (number of R1 universities, prior NIH funding), and socioeconomic factors (median income, healthcare expenditure per capita)? Additionally, does funding allocation show temporal lag responses to increasing disease burden, and do geographic regions exhibit funding disparities independent of burden?

### Python Libraries

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
pd.set_option('display.max_columns', None)

import matplotlib.pyplot as plt
import numpy as np

import glob

from sklearn.preprocessing import StandardScaler

## Load CDC Data

In [ ]:
final_df = None

for f in glob.glob("CDC data (2018-2023)/*"):
    print(f)
    df = pd.read_csv(f, quotechar='"', escapechar='\\', engine='python')
    df = df[~df['Year'].isna()]
    df['Year'] = df['Year'].astype(int)
    df['Year Code'] = df['Year Code'].astype(int)
    df['State Code'] = df['State Code'].astype(int)
    df['County Code'] = df['County Code'].astype(int)
    df['Deaths'] = df['Deaths'].astype(int)

    if final_df is None:
        final_df = df
    else:
        final_df = pd.concat([final_df, df])
final_df.head()

In [ ]:
final_df[["State", "Year"]].groupby("State").nunique()

### Definitions of CDC WONDER Columns

* **Notes** – Additional comments or flags attached to the record.
* **Year** – The calendar year of the death data.
* **Year Code** – Numeric code corresponding to the calendar year.
* **State** – U.S. state in which the deaths occurred.
* **State Code** – Numeric FIPS code for the state.
* **County** – County name and state abbreviation for the death record.
* **County Code** – Numeric FIPS code for the county.
* **Five-Year Age Groups** – Age group category in five-year intervals.
* **Five-Year Age Groups Code** – Numeric or short code representing the age group.
* **Sex** – Reported sex of the decedent.
* **Sex Code** – Abbreviated sex identifier (e.g., M, F).
* **Deaths** – Number of deaths recorded for the demographic group.
* **Population** – Population estimate for the group used to calculate rates.
* **Crude Rate** – Deaths per population without age adjustment.
* **% of Total Deaths** – Percentage of all deaths accounted for by this group.

### Load Income and Population Data

In [ ]:
per_capita_income = pd.read_csv("acs_income.csv")

pop = pd.read_csv("acs_population.csv")

median_income = pd.read_csv("acs_median_income.csv")

### Plot Missing Values

In [ ]:
def plot_missing_percentage(df):
    """
    Plots percentage of missing values for each column.

    Parameters
    ----------
    df : pandas.DataFrame
        Input DataFrame
    """

    for c in df.columns:
        df[c] = df[c].replace("Not Applicable", None, regex = False)

    # Calculate missing percentage per column
    missing_percent = (df.isnull().sum() / len(df)) * 100

    # Plot
    plt.figure(figsize=(12, 6))
    bars = plt.bar(missing_percent.index, missing_percent.values)

    plt.ylabel("Percentage of Missing Values (%)")
    plt.title("Missing Values Percentage by Column")
    plt.xticks(rotation=90)

    # Add percentage labels on each bar
    for bar, pct in zip(bars, missing_percent.values):
        plt.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height(),
            f"{pct:.1f}%",
            ha='center',
            va='bottom',
            fontsize=9
        )

    plt.tight_layout()
    plt.show()

In [ ]:
plot_missing_percentage(final_df)

### What Was the Problem? --> Missing Population!!!

The core issue was that the **Population column contained mixed data types**, including valid numbers and string placeholders such as `"Not Applicable"` and `"Not Available"`. This prevented direct numerical analysis and aggregation.

---

### How we solved the problem?

1. **Standardize percentage values**
   The `% of Total Deaths` column originally contained percentage symbols and string values. These were removed and converted to numeric floats so the column could be used in mathematical calculations.

2. **Derive missing population values (Not Applicable)**
   For rows where `Population` was marked as `"Not Applicable"`, population was calculated using the formula:
   
   $$ \text{Population} = \frac{\text{Deaths}}{\% \text{ of Total Deaths}}$$
   
   This uses available death counts and relative contribution to total deaths to infer population size.

3. **Prepare population values for aggregation**
   A temporary numeric column (`Population_A`) was created by coercing population values to numeric, allowing aggregation while safely handling non-numeric entries.

4. **Impute remaining missing values (Not Available)**
   For rows where population was `"Not Available"`, values were imputed using the **mean population within the same Year, State, County, and Sex group**, preserving demographic and geographic consistency.

5. **Finalize and clean the dataset**
   The population column was converted to integers, and the temporary helper column was removed to leave a clean, analysis-ready dataset.

---

In [ ]:
# 1. Convert % column to numeric once
final_df['pct_total_deaths'] = (
    final_df['% of Total Deaths']
    .str.rstrip('%')
    .astype(float)
)

# 2. Convert Population to numeric (NaN for non-numeric)
final_df['Population_num'] = pd.to_numeric(final_df['Population'], errors='coerce')

# 3. Calculate Population where value is "Not Applicable"
mask_na = final_df['Population'].isna()

final_df.loc[mask_na, 'Population_num'] = (
    final_df.loc[mask_na, 'Deaths'] /
    final_df.loc[mask_na, 'pct_total_deaths']
)

# 4. Fill remaining missing Population using group mean
group_mean = (
    final_df
    .groupby(['Year', 'State', 'County', 'Sex'])['Population_num']
    .transform('mean').round()
)

final_df['Population_num'] = final_df['Population_num'].fillna(group_mean)

# 5. Finalize Population column
final_df['Population'] = final_df['Population_num'].round().astype('Int64')

# 6. Cleanup helper columns
final_df.drop(columns=['Population_num', 'pct_total_deaths'], inplace=True)

In [ ]:
final_df[["State", "Year"]].groupby("State").nunique()

### Save preprocessed data

In [ ]:
final_df.to_csv("cdc_wonder_data_cleaned.csv", index=False)

### Filter to 2018-2023

In [ ]:
print(f"Total rows = {final_df.shape[0]}")
final_df = final_df[(final_df['Year'] >= 2018) & (final_df['Year'] <= 2023)]
print(f"Total rows from 2018 to 2023 = {final_df.shape[0]}")


    ╔════════════════════════════════════════════════════════════════════════╗
    ║                                                                        ║
    ║   ALZHEIMER'S RESEARCH FUNDING ANALYSIS                                ║
    ║   Feature Engineering & Dataset Aggregation Pipeline                   ║
    ║                                                                        ║
    ║   Research Question:                                                   ║
    ║   How is NIH funding associated with Alzheimer's burden at the         ║
    ║   state level (2018-2023), controlling for research capacity and       ║
    ║   socioeconomic factors, with temporal lag analysis?                   ║
    ║                                                                        ║
    ╚════════════════════════════════════════════════════════════════════════╝


### Disease Burden Features

In [ ]:
def create_disease_burden_features(df):
    """
    Category A: Disease Burden Features

    Creates mortality rates, death counts, and temporal trends that represent
    the severity and trajectory of Alzheimer's disease in each state.
    These are critical for measuring need and should theoretically drive funding.
    """
    print("\n[CDC] Creating Disease Burden Features...")

    # A1. Mortality Metrics
    state_year = df.groupby(['State', 'Year']).agg({
        'Deaths': 'sum',
    }).reset_index()

    state_year = state_year.merge(pop, on = ['State', 'Year'], how = "left")

    # Total deaths
    state_year['Total_Deaths_Annual'] = state_year['Deaths']

    # Mortality rate per 100k
    state_year['Mortality_Rate_Per_100k'] = (
        state_year['Deaths'] / state_year['Population'] * 100000
    )

    # Calculate mortality rank within each year
    state_year['Mortality_Rank_National'] = state_year.groupby('Year')[
        'Mortality_Rate_Per_100k'].rank(ascending=False, method='min')

    state_year['Mortality_Percentile'] = state_year.groupby('Year')[
        'Mortality_Rate_Per_100k'].rank(pct=True) * 100

    print(f"   ✓ Created 6 mortality metrics")

    # A2. Temporal Trends
    state_year = state_year.sort_values(['State', 'Year'])

    # Year-over-year changes
    state_year['Mortality_YoY_Change_Pct'] = state_year.groupby('State')[
        'Mortality_Rate_Per_100k'].pct_change() * 100

    state_year['Deaths_YoY_Change_Pct'] = state_year.groupby('State')[
        'Deaths'].pct_change() * 100

    # 3-year rolling average
    state_year['Mortality_3Yr_Rolling_Avg'] = state_year.groupby('State')[
        'Mortality_Rate_Per_100k'].transform(lambda x: x.rolling(3, min_periods=1).mean())

    # Mortality trend (slope over available years)
    def calculate_trend_slope(group):
        if len(group) < 3:
            return np.nan
        x = np.arange(len(group))
        y = group.values
        if np.all(np.isnan(y)):
            return np.nan
        slope, _ = np.polyfit(x[~np.isnan(y)], y[~np.isnan(y)], 1) if np.sum(~np.isnan(y)) >= 2 else (np.nan, np.nan)
        return slope

    state_year['Mortality_Trend_Slope'] = state_year.groupby('State')[
        'Mortality_Rate_Per_100k'].transform(calculate_trend_slope)

    # Cumulative deaths
    state_year['Deaths_Cumulative_2018'] = state_year.groupby('State')[
        'Deaths'].cumsum()

    # Volatility (rolling std)
    state_year['Mortality_Volatility'] = state_year.groupby('State')[
        'Mortality_Rate_Per_100k'].transform(lambda x: x.rolling(3, min_periods=1).std())

    print(f"   ✓ Created 6 temporal trend features")

    # A3. Geographic Distribution (county-level)
    if 'County' in df.columns:
        county_stats = df.groupby(['State', 'Year', 'County']).agg({
            'Deaths': 'sum'
        }).reset_index()

        # Top 3 counties' share of deaths
        def top3_concentration(group):
            if len(group) < 1:
                return np.nan
            top3 = group.nlargest(3, 'Deaths')['Deaths'].sum()
            total = group['Deaths'].sum()
            return (top3 / total * 100) if total > 0 else np.nan

        county_conc = county_stats.groupby(['State', 'Year']).apply(
            top3_concentration
        ).reset_index(name='County_Burden_Concentration')

        state_year = state_year.merge(county_conc, on=['State', 'Year'], how='left')

        # Number of high-burden counties (>100 deaths)
        high_burden = county_stats[county_stats['Deaths'] > 100].groupby(
            ['State', 'Year']).size().reset_index(name='High_Burden_County_Count')

        state_year = state_year.merge(high_burden, on=['State', 'Year'], how='left')
        state_year['High_Burden_County_Count'] = state_year['High_Burden_County_Count'].fillna(0)

        print(f"   ✓ Created 2 geographic distribution features")

    burden_features = state_year
    return burden_features

In [ ]:
final_df['Five-Year Age Groups'].unique()

### Demographic Features

In [ ]:
def create_demographic_features(df):
    """
    Category B: Demographic Features

    Captures at-risk populations (65+, 85+) which are critical for understanding
    future burden and research priorities. Age is the #1 risk factor for Alzheimer's.
    """
    print("\n[CDC] Creating Demographic Features...")

    # Define age groups
    age_65plus = ['65-69 years', '70-74 years', '75-79 years', '80-84 years',
                  '85-89 years', '90-94 years', '95-99 years', '100+ years']

    age_85plus = ['85-89 years', '90-94 years', '95-99 years', '100+ years']

    age_75_84 = ['75-79 years', '80-84 years']

    if 'Five-Year Age Groups' in df.columns:
        # Calculate by age group
        df_65plus = df[df['Five-Year Age Groups'].isin(age_65plus)]
        df_85plus = df[df['Five-Year Age Groups'].isin(age_85plus)]
        df_75_84 = df[df['Five-Year Age Groups'].isin(age_75_84)]

        pop_data = df[['State', 'Year']].drop_duplicates(subset = ['State', 'Year'])
        pop_data = pop_data.merge(pop, on = ['State', 'Year'], how = 'left')

        pop_data.columns = ['State', 'Year', 'Total_Population']

        # 65+ population
        pop_65 = df_65plus.groupby(['State', 'Year'])['Population'].sum().reset_index()
        pop_65.columns = ['State', 'Year', 'Population_65plus']

        # 85+ population
        pop_85 = df_85plus.groupby(['State', 'Year'])['Population'].sum().reset_index()
        pop_85.columns = ['State', 'Year', 'Population_85plus']

        # 75-84 population
        pop_75 = df_75_84.groupby(['State', 'Year'])['Population'].sum().reset_index()
        pop_75.columns = ['State', 'Year', 'Population_75_84']

        # Merge all
        pop_data = pop_data.merge(pop_65, on=['State', 'Year'], how='left')
        pop_data = pop_data.merge(pop_85, on=['State', 'Year'], how='left')
        pop_data = pop_data.merge(pop_75, on=['State', 'Year'], how='left')

    # Calculate percentages
    pop_data['Pct_Population_65plus'] = (
        pop_data['Population_65plus'] / pop_data['Total_Population'] * 100
    )
    pop_data['Pct_Population_85plus'] = (
        pop_data['Population_85plus'] / pop_data['Total_Population'] * 100
    )
    pop_data['Pct_Population_75_84'] = (
        pop_data['Population_75_84'] / pop_data['Total_Population'] * 100
    )

    # Elderly dependency ratio (65+ / 18-64)
    # Assuming working age is ~62% of population (US average)
    pop_data['Working_Age_Population'] = pop_data['Total_Population'] * 0.62
    pop_data['Elderly_Dependency_Ratio'] = (
        pop_data['Population_65plus'] / pop_data['Working_Age_Population'] * 100
    )

    # Very old ratio (85+ as % of 65+)
    pop_data['Very_Old_Ratio'] = (
        pop_data['Population_85plus'] / pop_data['Population_65plus'] * 100
    )

    # YoY growth rates
    pop_data = pop_data.sort_values(['State', 'Year'])
    pop_data['Age_65plus_YoY_Growth'] = pop_data.groupby('State')[
        'Population_65plus'].pct_change() * 100
    pop_data['Age_85plus_YoY_Growth'] = pop_data.groupby('State')[
        'Population_85plus'].pct_change() * 100

    print(f"   ✓ Created {len([c for c in pop_data.columns if c not in ['State', 'Year']])} demographic features")

    demographic_features = pop_data
    return demographic_features

In [ ]:
create_demographic_features(final_df).head(30)

### Sex-Specific Features

In [ ]:
def create_sex_specific_features(df):
    """
    Create sex-specific mortality patterns

    Women have higher Alzheimer's rates and live longer, affecting burden patterns.
    """
    print("\n[CDC] Creating Sex-Specific Features...")

    if 'Sex' not in df.columns:
        print("   ⚠ Sex column not found, skipping sex-specific features")
        return self

    sex_data = df.groupby(['State', 'Year', 'Sex']).agg({
        'Deaths': 'sum'
    }).reset_index()

    sex_data = sex_data.merge(pop, on = ['State', 'Year'], how = 'left')

    sex_data['Mortality_Rate'] = sex_data['Deaths'] / sex_data['Population'] * 100000

    # Pivot to get male/female rates
    sex_pivot = sex_data.pivot_table(
        index=['State', 'Year'],
        columns='Sex',
        values=['Deaths', 'Mortality_Rate'],
        aggfunc='sum'
    ).reset_index()

    # Flatten column names
    sex_pivot.columns = ['_'.join(col).strip('_') for col in sex_pivot.columns.values]

    # Calculate ratios
    if 'Deaths_Female' in sex_pivot.columns and 'Deaths_Male' in sex_pivot.columns:
        sex_pivot['Female_Male_Death_Ratio'] = (
            sex_pivot['Deaths_Female'] / sex_pivot['Deaths_Male']
        )
        sex_pivot['Female_Pct_Of_Deaths'] = (
            sex_pivot['Deaths_Female'] /
            (sex_pivot['Deaths_Female'] + sex_pivot['Deaths_Male']) * 100
        )

    print(f"   ✓ Created sex-specific features")

    sex_features = sex_pivot
    return sex_features

### Extract All CDC Features

In [ ]:
def get_all_cdc_features(df):

    burden_features = create_disease_burden_features(df)
    demographic_features = create_demographic_features(df)
    sex_features = create_sex_specific_features(df)

    """Merge all CDC features into one dataset"""
    print("\n[CDC] Merging all features...")

    # Start with burden features
    final = burden_features.copy()

    # Merge demographics
    final = final.merge(demographic_features, on=['State', 'Year'], how='left')

    # Merge sex features
    final = final.merge(sex_features, on=['State', 'Year'], how='left')

    # Add state code mapping
    state_to_code = {
        'Alabama': 'AL', 'Alaska': 'AK', 'Arizona': 'AZ', 'Arkansas': 'AR',
        'California': 'CA', 'Colorado': 'CO', 'Connecticut': 'CT', 'Delaware': 'DE',
        'Florida': 'FL', 'Georgia': 'GA', 'Hawaii': 'HI', 'Idaho': 'ID',
        'Illinois': 'IL', 'Indiana': 'IN', 'Iowa': 'IA', 'Kansas': 'KS',
        'Kentucky': 'KY', 'Louisiana': 'LA', 'Maine': 'ME', 'Maryland': 'MD',
        'Massachusetts': 'MA', 'Michigan': 'MI', 'Minnesota': 'MN', 'Mississippi': 'MS',
        'Missouri': 'MO', 'Montana': 'MT', 'Nebraska': 'NE', 'Nevada': 'NV',
        'New Hampshire': 'NH', 'New Jersey': 'NJ', 'New Mexico': 'NM', 'New York': 'NY',
        'North Carolina': 'NC', 'North Dakota': 'ND', 'Ohio': 'OH', 'Oklahoma': 'OK',
        'Oregon': 'OR', 'Pennsylvania': 'PA', 'Rhode Island': 'RI', 'South Carolina': 'SC',
        'South Dakota': 'SD', 'Tennessee': 'TN', 'Texas': 'TX', 'Utah': 'UT',
        'Vermont': 'VT', 'Virginia': 'VA', 'Washington': 'WA', 'West Virginia': 'WV',
        'Wisconsin': 'WI', 'Wyoming': 'WY', 'District of Columbia': 'DC'
    }

    if 'State Code' not in final.columns:
        final['State_Code'] = final['State'].map(state_to_code)
    else:
        final['State_Code'] = final['State Code']

    print(f"   ✓ Final CDC dataset: {final.shape}")
    print(f"   ✓ Features: {len(final.columns)}")
    print(f"   ✓ States: {final['State'].nunique()}")
    print(f"   ✓ Years: {final['Year'].nunique()}")

    return final

In [ ]:
final_cdc_df = get_all_cdc_features(final_df)

final_cdc_df.head()

In [ ]:
final_cdc_df[["State", "Year"]].groupby("State").nunique()

## NIH Data

In [ ]:
final_df_nih = None

for f in glob.glob("NIH data (2018-2023)/*"):
    print(f)
    df = pd.read_csv(f, engine='python')
    if final_df_nih is None:
        final_df_nih = df
    else:
        final_df_nih = pd.concat([final_df_nih, df])
final_df_nih.head()

### Data Definitions for Each Column

* **NIH Spending Categorization** – NIH’s thematic category assigned to the funded project.
* **Project Terms** – Keywords describing project topics, methods, populations, or diseases.
* **Project Title** – The official title of the NIH-funded research project.
* **Public Health Relevance** – Brief statement explaining the project’s significance to public health.
* **Administering IC** – NIH Institute or Center responsible for managing the award.
* **Application ID** – Internal NIH identifier for the submitted grant application.
* **Award Notice Date** – Date on which the funding award was officially issued.
* **Opportunity Number** – Funding opportunity announcement (FOA) under which the grant was submitted.
* **Project Number** – The unique NIH grant number assigned to the project.
* **Type** – Code indicating new, renewal, supplemental, or other award type.
* **Activity** – NIH activity code (e.g., R01, P30) describing the funding mechanism.
* **IC** – Abbreviation for the awarding NIH Institute or Center.
* **Serial Number** – Portion of the project number uniquely identifying the grant.
* **Support Year** – The specific budget year of the award.
* **Suffix** – Additional identifier marking supplements or revisions.
* **Program Official Information** – Name of the NIH program officer overseeing the grant.
* **Project Start Date** – Official start date of the funded project period.
* **Project End Date** – Official end date of the funded project period.
* **Study Section** – Peer review panel that evaluated the grant application.
* **Subproject Number** – Identifier for a sub-award or project component, if applicable.
* **Contact PI Person ID** – Internal NIH person ID for the Contact Principal Investigator.
* **Contact PI / Project Leader** – Name of the main Principal Investigator or project leader.
* **Other PI or Project Leader(s)** – Names of additional investigators on multi-PI awards.
* **Congressional District** – U.S. congressional district of the grantee institution.
* **Department** – Academic or organizational department conducting the research.
* **Primary DUNS** – Legacy DUNS number of the primary recipient institution.
* **Primary UEI** – Unique Entity Identifier for the primary recipient institution.
* **DUNS Number** – DUNS number associated with the award record.
* **UEI** – UEI associated with the award record.
* **FIPS** – U.S. county FIPS code for the institution’s location.
* **Latitude** – Latitude coordinate of the institution.
* **Longitude** – Longitude coordinate of the institution.
* **Organization ID (IPF)** – Institutional Profile File ID assigned by NIH.
* **Organization Name** – Name of the recipient organization.
* **Organization City** – City where the organization is located.
* **Organization State** – State or territory of the organization.
* **Organization Type** – Category describing the type of organization (e.g., academic, hospital).
* **Organization Zip** – ZIP code for the organization’s address.
* **Organization Country** – Country of the recipient organization.
* **ARRA Indicator** – Flag showing if the award was funded under ARRA stimulus funds.
* **Budget Start Date** – Start date of the specific budget period.
* **Budget End Date** – End date of the specific budget period.
* **Post Award Action Type** – Type of administrative action taken post-award, if any.
* **Assistance Listing Number** – Federal program number (formerly CFDA) for the grant.
* **Funding Mechanism** – Category of NIH funding (e.g., Non-SBIR/STTR).
* **Fiscal Year** – Federal fiscal year in which the award was funded.
* **Total Cost** – Total cost of the award for the given year (direct + indirect).
* **Total Cost (Sub Projects)** – Total cost for subprojects within multi-component awards.
* **Funding IC(s)** – NIH Institutes or Centers contributing funds.
* **Direct Cost IC** – Direct costs charged to the funding Institute/Center.
* **Indirect Cost IC** – Indirect costs charged to the funding Institute/Center.
* **NIH COVID-19 Response** – Indicator showing relevance to COVID-19 response funding.
* **Total Cost IC** – Total costs attributed specifically to the funding Institute/Center.

In [ ]:
final_df_nih[["ORG_STATE", "FY"]].groupby("ORG_STATE").nunique()

In [ ]:
# Convert to numeric
final_df_nih['FY'] = pd.to_numeric(final_df_nih['FY'], errors='coerce')
final_df_nih['TOTAL_COST'] = pd.to_numeric(final_df_nih['TOTAL_COST'], errors='coerce')
final_df_nih['DIRECT_COST_AMT'] = pd.to_numeric(final_df_nih['DIRECT_COST_AMT'], errors='coerce')
final_df_nih['INDIRECT_COST_AMT'] = pd.to_numeric(final_df_nih['INDIRECT_COST_AMT'], errors='coerce')

### Filter to 2019-2023

In [ ]:
final_df_nih = final_df_nih[(final_df_nih['FY'] >= 2019) & (final_df_nih['FY'] <= 2023)]

In [ ]:
final_df_nih[["ORG_STATE", "FY"]].groupby("ORG_STATE").nunique()

### Overall Funding (State + Year)

In [ ]:
# Basic funding metrics
overall_funding = final_df_nih.groupby(['ORG_STATE', 'FY']).agg({
    'TOTAL_COST': ['sum', 'mean'],
    'DIRECT_COST_AMT': 'sum',
    'INDIRECT_COST_AMT': 'sum',
    'APPLICATION_ID': 'count'
}).round(0).astype(int).reset_index()

overall_funding.columns = ["State", "Year", "Overall_Total_Funding", "Overall_Avg_Funding",
                           "Overall_Total_Direct_Funding", "Overall_Total_Indirect_Funding", "Overall_Total_Projects"]

overall_funding.head()

### Only Select Data of US

In [ ]:
final_df_nih = final_df_nih[final_df_nih["ORG_COUNTRY"] == "UNITED STATES"]

In [ ]:
final_df_nih[["ORG_STATE", "FY"]].groupby("ORG_STATE").nunique()

### Only Select Alzheimer Related Funding

In [ ]:
final_df_nih = final_df_nih[final_df_nih["PROJECT_TERMS"].fillna("").str.contains("Alzheimer")]

In [ ]:
final_df_nih[["ORG_STATE", "FY"]].groupby("ORG_STATE").nunique()

### Some Valid Zip Codes i.e of length 9

In [ ]:
final_df_nih = final_df_nih[final_df_nih["ORG_ZIPCODE"].str.len() == 9]

### Extract Counties From Zip Codes

In [ ]:
import zipcodes

# Extract 5-digit ZIP codes safely
zip_series = (
    final_df_nih['ORG_ZIPCODE']
    .astype(str)
    .str[:5]
)

# Cache for ZIP → county lookup
zip_to_county = {}

def get_county(zip_code):
    if zip_code not in zip_to_county:
        details = zipcodes.matching(zip_code)
        zip_to_county[zip_code] = details[0]['county'] if details else None
    return zip_to_county[zip_code]

# Apply function vectorized over the column
final_df_nih['County'] = zip_series.map(get_county)

In [ ]:
plot_missing_percentage(final_df_nih)

In [ ]:
final_df_nih[final_df_nih["ORG_STATE"] == "AK"]

In [ ]:
final_df_nih[final_df_nih["ORG_STATE"] == "AL"]['FY'].unique()

In [ ]:
final_df_nih.to_csv("nih_funding_by_state_cleaned.csv", index=False)

### Funding Allocation Features

In [ ]:
def create_funding_features(df):
    """
    Category D: Funding Allocation Features

    Primary outcome variables showing how NIH distributes Alzheimer's research
    funding across states. Critical for testing funding-burden alignment.
    """
    print("\n[NIH] Creating Funding Features...")

    # D1. Basic funding metrics
    funding = df.groupby(['ORG_STATE', 'FY']).agg({
        'TOTAL_COST': ['sum', 'mean', 'median', 'std'],
        'DIRECT_COST_AMT': 'sum',
        'INDIRECT_COST_AMT': 'sum',
        'CORE_PROJECT_NUM': 'nunique',
        'APPLICATION_ID': 'count'
    }).reset_index()

    # Flatten column names
    funding.columns = ['State_Code', 'Year',
                      'Total_Funding_Annual', 'Mean_Award_Size',
                      'Median_Award_Size', 'Award_Size_StdDev',
                      'Total_Direct_Cost', 'Total_Indirect_Cost',
                      'Number_Of_Projects', 'Number_Of_Applications']

    # Convert to millions
    funding['Total_Funding_Millions'] = funding['Total_Funding_Annual'] / 1e6
    funding['Mean_Award_Millions'] = funding['Mean_Award_Size'] / 1e6

    # Indirect cost ratio
    funding['Indirect_Cost_Ratio'] = (
        funding['Total_Indirect_Cost'] / funding['Total_Direct_Cost']
    )

    print(f"   ✓ Created 11 basic funding metrics")

    # D2. Temporal funding patterns
    funding = funding.sort_values(['State_Code', 'Year'])

    funding['Funding_YoY_Change_Pct'] = funding.groupby('State_Code')[
        'Total_Funding_Annual'].pct_change() * 100

    funding['Projects_YoY_Change'] = funding.groupby('State_Code')[
        'Number_Of_Projects'].diff()

    funding['Funding_3Yr_Rolling_Avg'] = funding.groupby('State_Code')[
        'Total_Funding_Annual'].transform(lambda x: x.rolling(3, min_periods=1).mean())

    # Funding trend slope
    def calc_slope(group):
        if len(group) < 3:
            return np.nan
        x = np.arange(len(group))
        y = group.values
        if np.all(np.isnan(y)):
            return np.nan
        valid = ~np.isnan(y)
        if np.sum(valid) < 2:
            return np.nan
        slope, _ = np.polyfit(x[valid], y[valid], 1)
        return slope

    funding['Funding_Trend_Slope'] = funding.groupby(['State_Code'])[
        'Total_Funding_Annual'].transform(calc_slope)

    # Cumulative funding
    funding['Funding_Cumulative_2018'] = funding.groupby(['State_Code'])[
        'Total_Funding_Annual'].cumsum()

    # Volatility
    funding['Funding_Volatility'] = funding.groupby(['State_Code'])[
        'Total_Funding_Annual'].transform(lambda x: x.rolling(3, min_periods=1).std())

    print(f"   ✓ Created 6 temporal funding features")

    funding_features = funding
    return funding_features

### Research Infrastructure Features

In [ ]:
def create_infrastructure_features(df):
    """
    Category E: Research Infrastructure Features

    Control variables capturing research capacity that may explain funding
    independent of disease burden (e.g., presence of major universities).
    """
    print("\n[NIH] Creating Research Infrastructure Features...")

    infra = df.groupby(['ORG_STATE', 'FY']).agg({
        'ORG_NAME': 'nunique',
        'PI_IDS': 'nunique',
        'CORE_PROJECT_NUM': 'nunique'
    }).reset_index()

    infra.columns = ['State_Code', 'Year', 'Number_Of_Institutions',
                    'Number_Of_PIs', 'Number_Of_Unique_Projects']

    # Count institution types if available
    if 'ED_INST_TYPE' in df.columns:
        # Universities
        uni_count = df[
            df['ED_INST_TYPE'].str.contains('UNIVERSITY|COLLEGE', na=False, case=False)
        ].groupby(['ORG_STATE', 'FY'])['ORG_NAME'].nunique().reset_index()
        uni_count.columns = ['State_Code', 'Year', 'Number_Of_Universities']

        infra = infra.merge(uni_count, on=['State_Code', 'Year'], how='left')
        infra['Number_Of_Universities'] = infra['Number_Of_Universities'].fillna(0)

        # SCHOOLS
        uni_count = df[
            df['ED_INST_TYPE'].str.contains('SCHOOLS', na=False, case=False)
        ].groupby(['ORG_STATE', 'FY'])['ORG_NAME'].nunique().reset_index()
        uni_count.columns = ['State_Code', 'Year', 'Number_Of_Schools']

        infra = infra.merge(uni_count, on=['State_Code', 'Year'], how='left')
        infra['Number_Of_Schools'] = infra['Number_Of_Schools'].fillna(0)

        # Medical centers
        med_count = df[
            df['ED_INST_TYPE'].str.contains('HOSPITAL|MEDICAL|CENTERS|UNIT', na=False, case=False)
        ].groupby(['ORG_STATE', 'FY'])['ORG_NAME'].nunique().reset_index()
        med_count.columns = ['State_Code', 'Year', 'Number_Of_Medical_Centers']

        infra = infra.merge(med_count, on=['State_Code', 'Year'], how='left')
        infra['Number_Of_Medical_Centers'] = infra['Number_Of_Medical_Centers'].fillna(0)

    # Calculate concentration (Herfindahl index approximation)
    # Top institution's share of funding
    top_share = df.groupby(['ORG_STATE', 'FY', 'ORG_NAME'])['TOTAL_COST'].sum().reset_index()
    top_share = top_share.sort_values('TOTAL_COST', ascending=False)

    def calc_top_share(group):
        total = group['TOTAL_COST'].sum()
        if total == 0:
            return np.nan
        return group.iloc[0]['TOTAL_COST'] / total * 100

    top_inst = top_share.groupby(['ORG_STATE', 'FY']).apply(calc_top_share).reset_index()
    top_inst.columns = ['State_Code', 'Year', 'Top_Institution_Funding_Share']

    infra = infra.merge(top_inst, on=['State_Code', 'Year'], how='left')

    # PIs per institution
    infra['PIs_Per_Institution'] = (
        infra['Number_Of_PIs'] / infra['Number_Of_Institutions']
    )

    print(f"   ✓ Created infrastructure features")

    infrastructure_features = infra
    return infrastructure_features

### Extract NIH Features

In [ ]:
def get_all_nih_features(final_df_nih):

    funding_features = create_funding_features(final_df_nih)
    infrastructure_features = create_infrastructure_features(final_df_nih)

    """Merge all NIH features into one dataset"""
    print("\n[NIH] Merging all features...")

    # Start with funding features
    final = funding_features.copy()

    # Merge infrastructure
    final = final.merge(infrastructure_features,
                       on=['State_Code', 'Year'], how='left')

    print(f"   ✓ Final NIH dataset: {final.shape}")
    print(f"   ✓ Features: {len(final.columns)}")
    print(f"   ✓ States: {final['State_Code'].nunique()}")
    print(f"   ✓ Years: {final['Year'].nunique()}")

    return final

In [ ]:
final_df_nih = get_all_nih_features(final_df_nih)

### Merge NIH and CDC datasets

In [ ]:
def merge_datasets(cdc, nih):
    """
    Merge CDC and NIH data on State_Code and Year

    This creates the complete state-year panel dataset for analysis.
    """
    print("\n" + "="*80)
    print("MERGING CDC AND NIH DATASETS")
    print("="*80)

    print("\n[1/5] Merging datasets...")

    # Merge on State_Code and Year
    merged = cdc.merge(
        nih,
        on=['State_Code', 'Year'],
        how='inner',
        indicator=True
    )

    print(f"   ✓ Merged dataset shape: {merged.shape}")
    print(f"\nMerge Statistics:")
    print(merged['_merge'].value_counts())

    # Keep only records with both CDC and NIH data for main analysis
    complete_data = merged[merged['_merge'] == 'both'].copy()
    complete_data = complete_data.drop('_merge', axis=1)

    print(f"\n   ✓ Complete cases (both CDC & NIH): {len(complete_data)}")
    print(f"   ✓ States with complete data: {complete_data['State_Code'].nunique()}")

    merged_data = complete_data
    return merged_data

In [ ]:
merged_data = merge_datasets(final_cdc_df, final_df_nih)

In [ ]:
merged_data.head()

### Create lagged features for temporal analysis

In [ ]:
def create_lagged_features(merged_data, lags=[1, 2, 3]):
    """
    Create lagged features for temporal analysis

    Lagged features allow testing whether funding responds to past burden
    (lag burden → predict current funding) or vice versa.
    Essential for answering temporal lag questions in research question.
    """
    print("\n[2/5] Creating lagged features...")

    df = merged_data.copy()
    df = df.sort_values(['State_Code', 'Year'])

    lag_features = [
        'Mortality_Rate_Per_100k',
        'Total_Deaths_Annual',
        'Population_65plus',
        'Population_85plus',
        'Total_Funding_Annual',
        'Number_Of_Projects'
    ]

    for feature in lag_features:
        if feature in df.columns:
            for lag in lags:
                df[f'{feature}_Lag{lag}'] = df.groupby('State_Code')[feature].shift(lag)

    created_lags = [c for c in df.columns if '_Lag' in c]
    print(f"   ✓ Created {len(created_lags)} lagged features")

    merged_data = df
    return merged_data

In [ ]:
merged_data = create_lagged_features(merged_data)

In [ ]:
merged_data.head()

In [ ]:
merged_data = merged_data.drop(columns = ['Population']).merge(pop, on = ['State', 'Year'], how = 'left')

### Create funding-to-burden ratio features

In [ ]:
def create_equity_features(merged_data):
    """
    Create funding-to-burden ratio features

    These ratios measure equity: how funding aligns with need.
    Critical for identifying underserved states in research question.
    """
    print("\n[3/5] Creating equity/ratio features...")

    df = merged_data.copy()

    # Funding per death
    df['Funding_Per_Death'] = df['Total_Funding_Annual'] / df['Total_Deaths_Annual']

    # Funding per 65+ person
    df['Funding_Per_65plus'] = df['Total_Funding_Annual'] / df['Population_65plus']

    # Funding per capita (total population)
    df['Funding_Per_Capita'] = df['Total_Funding_Annual'] / df['Total_Population']

    # Projects per million population
    df['Projects_Per_Million_Pop'] = (
        df['Number_Of_Projects'] / df['Total_Population'] * 1e6
    )

    # Funding per mortality unit (per 100k rate)
    df['Funding_Per_Mortality_Unit'] = (
        df['Total_Funding_Annual'] / df['Mortality_Rate_Per_100k']
    )

    # Deaths per elderly (disease prevalence in at-risk group)
    df['Deaths_Per_100k_Elderly'] = (
        df['Total_Deaths_Annual'] / df['Population_65plus'] * 100000
    )

    equity_features = [c for c in df.columns if 'Per_' in c or 'Ratio' in c]
    print(f"   ✓ Created {len([c for c in equity_features if c not in merged_data.columns])} equity features")

    merged_data = df
    return merged_data

In [ ]:
merged_data = create_equity_features(merged_data)

### Create composite scores combining multiple indicators

In [ ]:
def create_composite_indices(merged_data):
    """
    Create composite scores combining multiple indicators

    Composite indices summarize complex constructs (burden, capacity, equity)
    into single interpretable scores for analysis.
    """
    print("\n[4/5] Creating composite indices...")

    df = merged_data.copy()

    # Standardize for composites
    scaler = StandardScaler()

    # Overall Burden Index (higher = worse burden)
    burden_vars = ['Mortality_Rate_Per_100k', 'Total_Deaths_Annual', 'Pct_Population_65plus']
    available_burden = [v for v in burden_vars if v in df.columns]

    if len(available_burden) >= 2:
        burden_scaled = scaler.fit_transform(df[available_burden].fillna(0))
        df['Overall_Burden_Index'] = burden_scaled.mean(axis=1)

    # Research Capacity Index
    capacity_vars = ['Number_Of_Institutions', 'Number_Of_PIs', 'Number_Of_Projects']
    available_capacity = [v for v in capacity_vars if v in df.columns]

    if len(available_capacity) >= 2:
        capacity_scaled = scaler.fit_transform(df[available_capacity].fillna(0))
        df['Research_Capacity_Index'] = capacity_scaled.mean(axis=1)

    # Funding-Burden Alignment Score (how well funding matches burden)
    if 'Overall_Burden_Index' in df.columns:
        # Standardize funding
        if 'Total_Funding_Annual' in df.columns:
            funding_scaled = scaler.fit_transform(df[['Total_Funding_Annual']].fillna(0))
            # Alignment = correlation-like measure (both high or both low = good)
            df['Funding_Burden_Alignment'] = -(
                np.abs(funding_scaled.flatten() - df['Overall_Burden_Index'])
            )

    composite_cols = [c for c in df.columns if 'Index' in c or 'Alignment' in c]
    print(f"   ✓ Created {len([c for c in composite_cols if c not in merged_data.columns])} composite indices")

    merged_data = df
    return merged_data

In [ ]:
merged_data = create_composite_indices(merged_data)

### Add external socioeconomic and research capacity controls

In [ ]:
def add_external_control_variables(merged_data):
    """
    Add external socioeconomic and research capacity controls

    These control variables are essential for isolating the burden-funding
    relationship from confounders mentioned in research question.

    NOTE: Using synthetic data for demonstration. Replace with:
    - Census Bureau: income, population
    - BEA: state GDP
    - CMS: healthcare expenditure
    - Carnegie Classification: R1 universities
    """
    print("\n[5/5] Adding external control variables...")
    print("   [INFO] Using synthetic data - replace with real external sources")

    df = merged_data.copy()

    r1 = pd.read_csv("r1_institutes.csv")
    df = df.merge(r1.drop(columns = ['State_Code']), on = ['State', 'Year'], how = 'left')

    # Create time-varying controls

    df = df.merge(per_capita_income, on = ['State', 'Year'], how = 'left')
    df = df.rename(columns = {"Per Capita Income" : "Per_Capita_Income"})

    df = df.merge(median_income, on = ['State', 'Year'], how = 'left')
    df = df.rename(columns = {"Median Income" : "Median_Income"})

    # Prior NIH funding (cumulative or lagged)
    df = df.sort_values(['State', 'Year'])
    df['Prior_NIH_Funding_Total'] = df.groupby('State')[
        'Total_Funding_Annual'].shift(1).fillna(0)

    # Rural/urban indicator (synthetic)
    df['Rural_Population_Pct'] = np.random.uniform(10, 60, len(df))

    control_vars = [
        'Num_R1_Universities', 'Median_Income',
        'Per_Capita_Income', 'Prior_NIH_Funding_Total'
    ]
    print(f"   ✓ Added {len(control_vars)} control variables")
    print(f"   ✓ Controls: {control_vars}")

    merged_data = df
    return merged_data

In [ ]:
merged_data = add_external_control_variables(merged_data)

### Create regional groupings and features

In [ ]:
def create_regional_features(merged_data):
    """
    Create regional groupings and features

    Regional analysis tests whether geographic disparities exist
    independent of burden (part of research question).
    """
    print("\n[BONUS] Creating regional features...")

    df = merged_data.copy()

    # Define Census regions
    region_map = {
        # Northeast
        'CT': 'Northeast', 'ME': 'Northeast', 'MA': 'Northeast', 'NH': 'Northeast',
        'RI': 'Northeast', 'VT': 'Northeast', 'NJ': 'Northeast', 'NY': 'Northeast',
        'PA': 'Northeast',
        # Midwest
        'IL': 'Midwest', 'IN': 'Midwest', 'MI': 'Midwest', 'OH': 'Midwest',
        'WI': 'Midwest', 'IA': 'Midwest', 'KS': 'Midwest', 'MN': 'Midwest',
        'MO': 'Midwest', 'NE': 'Midwest', 'ND': 'Midwest', 'SD': 'Midwest',
        # South
        'DE': 'South', 'FL': 'South', 'GA': 'South', 'MD': 'South',
        'NC': 'South', 'SC': 'South', 'VA': 'South', 'WV': 'South',
        'AL': 'South', 'KY': 'South', 'MS': 'South', 'TN': 'South',
        'AR': 'South', 'LA': 'South', 'OK': 'South', 'TX': 'South',
        'DC': 'South',
        # West
        'AZ': 'West', 'CO': 'West', 'ID': 'West', 'MT': 'West',
        'NV': 'West', 'NM': 'West', 'UT': 'West', 'WY': 'West',
        'AK': 'West', 'CA': 'West', 'HI': 'West', 'OR': 'West', 'WA': 'West'
    }

    df['Region'] = df['State_Code'].map(region_map)

    # Regional averages for comparison
    regional_stats = df.groupby(['Region', 'Year']).agg({
        'Total_Funding_Annual': 'mean',
        'Mortality_Rate_Per_100k': 'mean',
        'Population_65plus': 'mean'
    }).reset_index()

    regional_stats.columns = ['Region', 'Year', 'Regional_Avg_Funding',
                              'Regional_Avg_Mortality', 'Regional_Avg_65plus_Pop']

    df = df.merge(regional_stats, on=['Region', 'Year'], how='left')

    # State deviation from regional average
    df['Funding_vs_Regional_Avg'] = (
        df['Total_Funding_Annual'] - df['Regional_Avg_Funding']
    )

    df['Mortality_vs_Regional_Avg'] = (
        df['Mortality_Rate_Per_100k'] - df['Regional_Avg_Mortality']
    )

    print(f"   ✓ Created regional features")
    print(f"   ✓ Regions: {df['Region'].unique()}")

    merged_data = df
    return merged_data

In [ ]:
merged_data = create_regional_features(merged_data)

### Final Data for Analysis

In [ ]:
def get_final_dataset(merged_data):
    """
    Return final aggregated dataset ready for analysis
    """
    print("\n" + "="*80)
    print("FINAL AGGREGATED DATASET")
    print("="*80)

    df = merged_data.copy()

    print(f"\n📊 Dataset Summary:")
    print(f"   Shape: {df.shape[0]} rows × {df.shape[1]} columns")
    print(f"   Time period: {df['Year'].min():.0f} - {df['Year'].max():.0f}")
    print(f"   States: {df['State_Code'].nunique()}")
    print(f"   State-year observations: {len(df)}")

    print(f"\n📈 Feature Categories:")
    burden_cols = [c for c in df.columns if any(x in c for x in ['Death', 'Mortality', 'Burden'])]
    demo_cols = [c for c in df.columns if any(x in c for x in ['Population', 'Age_', 'Pct_'])]
    funding_cols = [c for c in df.columns if any(x in c for x in ['Funding', 'Award', 'Cost'])]
    infra_cols = [c for c in df.columns if any(x in c for x in ['Institution', 'PI', 'Project', 'Grant'])]
    lag_cols = [c for c in df.columns if 'Lag' in c]
    equity_cols = [c for c in df.columns if 'Per_' in c or 'Ratio' in c]
    control_cols = [c for c in df.columns if any(x in c for x in ['GDP', 'Income', 'Healthcare', 'R1'])]

    print(f"   • Disease Burden: {len(burden_cols)} features")
    print(f"   • Demographics: {len(demo_cols)} features")
    print(f"   • Funding Outcomes: {len(funding_cols)} features")
    print(f"   • Infrastructure: {len(infra_cols)} features")
    print(f"   • Lagged Variables: {len(lag_cols)} features")
    print(f"   • Equity Ratios: {len(equity_cols)} features")
    print(f"   • Control Variables: {len(control_cols)} features")

    print(f"\n⚠️  Missing Data:")
    missing = df.isnull().sum()
    missing_pct = (missing / len(df) * 100).round(1)
    high_missing = missing_pct[missing_pct > 10].sort_values(ascending=False)

    if len(high_missing) > 0:
        print(f"   Columns with >10% missing:")
        for col, pct in high_missing.head(10).items():
            print(f"      • {col}: {pct}%")
    else:
        print(f"   ✓ No columns with >10% missing data")

    print(f"\n✅ Dataset ready for analysis!")

    return df

In [ ]:
final_dataset = get_final_dataset(merged_data)

In [ ]:
final_dataset.head()

In [ ]:
final_dataset[final_dataset['State'] == "Alabama"]

In [ ]:
# Define required years
required_years = [2019, 2020, 2021, 2022, 2023]

# Create a MultiIndex of all state-year combinations
states = final_dataset['State'].unique()

full_index = pd.MultiIndex.from_product([states, required_years], names=['State', 'Year'])

# Reindex the DataFrame to include all combinations
final_dataset = final_dataset.set_index(['State', 'Year']).reindex(full_index, fill_value=0).reset_index()

In [ ]:
final_dataset[["State", "Year"]].groupby("State").count()

In [ ]:
final_dataset.to_csv("final_dataset_for_analysis.csv", index=False)